# History

> Generic undo history stack for workflow operations.

In [ ]:
#| default_exp history

In [ ]:
#| export
from typing import Optional, Tuple, List, Dict, Any

## Constants

In [ ]:
#| export
DEFAULT_MAX_HISTORY_DEPTH = 50

## History Operations

Pure functions for managing an undo history stack. The stack stores opaque
snapshots — each workflow decides what to include in a snapshot (e.g., segments
and focused index for a card stack, relations and selected node for a graph editor).

In [ ]:
#| export
def push_history(
    history: List[Dict[str, Any]],  # Current history stack
    snapshot: Dict[str, Any],  # State snapshot to save
    max_depth: int = DEFAULT_MAX_HISTORY_DEPTH,  # Maximum history depth
) -> List[Dict[str, Any]]:  # Updated history stack
    """Push a state snapshot onto the history stack, enforcing max depth."""
    new_history = list(history)
    new_history.append(snapshot)
    if len(new_history) > max_depth:
        new_history = new_history[-max_depth:]
    return new_history

In [ ]:
#| export
def pop_history(
    history: List[Dict[str, Any]],  # Current history stack
) -> Optional[Tuple[Dict[str, Any], List[Dict[str, Any]]]]:  # (snapshot, updated_history) or None
    """Pop the most recent snapshot from the history stack."""
    if not history:
        return None
    new_history = list(history)
    snapshot = new_history.pop()
    return snapshot, new_history

## Tests

In [ ]:
# Push to empty history
h = push_history([], {"items": ["a"], "focus": 0})
assert len(h) == 1
assert h[0]["items"] == ["a"]
assert h[0]["focus"] == 0

# Push preserves existing entries
h = push_history(h, {"items": ["a", "b"], "focus": 1})
assert len(h) == 2
assert h[1]["focus"] == 1

# Max depth enforcement
h = push_history([], {"v": "old"}, max_depth=2)
h = push_history(h, {"v": "mid"}, max_depth=2)
h = push_history(h, {"v": "new"}, max_depth=2)
assert len(h) == 2
assert h[0]["v"] == "mid"
assert h[1]["v"] == "new"

# Original list is not mutated
original = [{"v": "x"}]
result = push_history(original, {"v": "y"})
assert len(original) == 1
assert len(result) == 2

print("push_history tests passed!")

push_history tests passed!


In [ ]:
# Pop from empty history returns None
assert pop_history([]) is None

# Pop returns last snapshot and updated history
h = [
    {"items": ["first"], "focus": 0},
    {"items": ["second"], "focus": 1},
]
snapshot, remaining = pop_history(h)
assert snapshot["items"] == ["second"]
assert snapshot["focus"] == 1
assert len(remaining) == 1

# Original list is not mutated
original = [{"v": "x"}]
snapshot, remaining = pop_history(original)
assert len(original) == 1
assert snapshot["v"] == "x"
assert len(remaining) == 0

# Arbitrary snapshot shapes work
h = push_history([], {"relations": [{"a": "b"}], "selected_node": "n1", "extra": 42})
snapshot, remaining = pop_history(h)
assert snapshot["relations"] == [{"a": "b"}]
assert snapshot["selected_node"] == "n1"
assert snapshot["extra"] == 42

print("pop_history tests passed!")

pop_history tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()